# Notebook 03 — RAG Evaluation Demo

## Project 01 — RAG Document Intelligence

This notebook demonstrates the evaluation workflow for the RAG system.

The evaluation pipeline is:

```text
golden questions
→ vector retrieval
→ retrieval evaluation
→ optional answer generation
→ answer evaluation
→ summary metrics
```

This notebook can run in two modes:

1. **Retrieval-only mode**: does not require an OpenAI API key.
2. **LLM-based mode**: requires `OPENAI_API_KEY`.

The default is retrieval-only mode for safety and reproducibility.


## 1. Setup

Run this notebook from either the project root or the `notebooks/` folder.

The cell below makes `src/` importable.


In [1]:
from pathlib import Path
import sys
import os
import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project root:", PROJECT_ROOT)
print("Source directory:", SRC_DIR)


Project root: C:\Users\aferr\Documents\GitHub\01-rag-document-intelligence
Source directory: C:\Users\aferr\Documents\GitHub\01-rag-document-intelligence\src


## 2. Check project paths

This notebook expects the vector store to already exist.

The vector store is generated by:

```text
notebooks/01_document_pipeline_demo.ipynb
```

Expected local folder:

```text
chroma_db/
```


In [2]:
from agro_rag.utils.paths import (
    ensure_project_directories,
    PROCESSED_DATA_DIR,
    REPORTS_DIR,
    CHROMA_DB_DIR,
)

ensure_project_directories()

print("Processed data directory:", PROCESSED_DATA_DIR)
print("Reports directory:", REPORTS_DIR)
print("ChromaDB directory:", CHROMA_DB_DIR)
print("ChromaDB exists:", CHROMA_DB_DIR.exists())
print("ChromaDB contains files:", any(CHROMA_DB_DIR.iterdir()) if CHROMA_DB_DIR.exists() else False)


Processed data directory: C:\Users\aferr\Documents\GitHub\01-rag-document-intelligence\data\processed
Reports directory: C:\Users\aferr\Documents\GitHub\01-rag-document-intelligence\reports
ChromaDB directory: C:\Users\aferr\Documents\GitHub\01-rag-document-intelligence\chroma_db
ChromaDB exists: True
ChromaDB contains files: True


## 3. Generate or load golden questions

The golden question set is used to evaluate:

- expected source retrieval;
- risk-screening behavior;
- limitation behavior;
- unsafe legal-overclaim avoidance.

Output:

```text
data/processed/rag_eval_questions.csv
```


In [3]:
from agro_rag.evaluation.golden_questions import (
    save_golden_questions,
    load_golden_questions,
)

golden_questions_path = PROCESSED_DATA_DIR / "rag_eval_questions.csv"

if golden_questions_path.exists():
    golden_questions = load_golden_questions(golden_questions_path)
    print("Loaded existing golden questions.")
else:
    golden_questions = save_golden_questions(golden_questions_path)
    print("Created golden questions.")

print("Golden questions:", len(golden_questions))
golden_questions.head()


Loaded existing golden questions.
Golden questions: 20


,question_id,question,expected_source,expected_theme,expected_metadata,expected_answer_type,unsafe_risk,notes
0,Q001,What does INPE / PRODES indicate about annual ...,INPE,deforestation,PRODES; annual monitoring,direct_answer,False,Should retrieve INPE or TerraBrasilis evidence...
1,Q002,What does MapBiomas indicate about land use an...,MapBiomas,land use and land cover,MapBiomas; land cover; land use,direct_answer,False,Should retrieve MapBiomas evidence about land-...
2,Q003,How can MapBiomas data help identify native ve...,MapBiomas,native vegetation,native vegetation; land-use transition,explanation,False,Should explain that MapBiomas supports land-co...
3,Q004,How can INPE and MapBiomas evidence complement...,MapBiomas; INPE,source comparison,PRODES; MapBiomas; land use; deforestation,comparison,False,Should compare official deforestation monitori...
4,Q005,What is the difference between deforestation e...,MapBiomas; INPE,methodology,deforestation; land-use transition,comparison,False,Should distinguish INPE deforestation monitori...


## 4. Inspect golden question coverage

The golden questions should cover:

- MapBiomas;
- INPE;
- source comparison;
- risk summaries;
- insufficient evidence cases;
- unsafe legal-overclaim cases.


In [4]:
golden_questions.groupby(
    ["expected_source", "expected_answer_type"],
    dropna=False,
).size().reset_index(name="n_questions")


,expected_source,expected_answer_type,n_questions
0,INPE,direct_answer,1
1,MapBiomas,direct_answer,1
2,MapBiomas,explanation,1
3,MapBiomas; INPE,comparison,2
4,MapBiomas; INPE,indicator_explanation,1
5,MapBiomas; INPE,limitations,1
6,MapBiomas; INPE,methodological_answer,1
7,MapBiomas; INPE,risk_summary,2
8,Project evaluation,methodological_answer,1
9,Project limitations,insufficient_evidence,1


In [5]:
golden_questions["unsafe_risk"].value_counts(dropna=False)


unsafe_risk
False    15
True      5
Name: count, dtype: int64

## 5. Confirm vector store readiness

Before running retrieval evaluation, the vector store must exist.

If the check fails, run Notebook 01 first:

```text
notebooks/01_document_pipeline_demo.ipynb
```


In [6]:
def chroma_is_ready(chroma_dir: Path) -> bool:
    return chroma_dir.exists() and any(chroma_dir.iterdir())

if chroma_is_ready(CHROMA_DB_DIR):
    print("Vector store is available.")
else:
    print("Vector store is not available.")
    print("Run notebooks/01_document_pipeline_demo.ipynb before full retrieval evaluation.")


Vector store is available.


## 6. Run retrieval evaluation

This step checks whether the retriever returns evidence from the expected source family.

Output:

```text
reports/rag_retrieval_evaluation.csv
```


In [7]:
from agro_rag.evaluation.evaluate_retrieval import (
    evaluate_retrieval_file,
    summarize_retrieval_evaluation,
)

retrieval_eval_path = REPORTS_DIR / "rag_retrieval_evaluation.csv"

if chroma_is_ready(CHROMA_DB_DIR):
    retrieval_eval = evaluate_retrieval_file(
        golden_questions_path=golden_questions_path,
        output_path=retrieval_eval_path,
        persist_directory=CHROMA_DB_DIR,
        collection_name="agro_environmental_documents",
        n_results=5,
    )

    retrieval_summary = summarize_retrieval_evaluation(retrieval_eval)
    print(retrieval_summary)
else:
    retrieval_eval = pd.DataFrame()
    retrieval_summary = {}
    print("Skipped retrieval evaluation because ChromaDB is not available.")

retrieval_eval.head()


C:\Users\aferr\Documents\GitHub\01-rag-document-intelligence\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████████████████| 103/103 [00:00<00:00, 3800.73it/s]


Retrieval evaluation summary
----------------------------
Questions: 20
Source hit rate@5: 0.900
Mean precision@5: 0.450
Mean recall@5: 0.900
{'n_questions': 20, 'source_hit_rate_at_k': 0.9, 'mean_precision_at_k': 0.45, 'mean_recall_at_k': 0.9}


,question_id,question,expected_source,expected_theme,expected_answer_type,unsafe_risk,n_results,source_hit_at_k,recall_at_k,precision_at_k,retrieved_sources,retrieved_documents,top_chunk_id,top_distance
0,Q001,What does INPE / PRODES indicate about annual ...,INPE,deforestation,direct_answer,False,5,1,1,0.333333,INPE; MapBiomas; Project limitations,synthetic_inpe_prodes_note; synthetic_mapbioma...,inpe__synthetic_inpe_prodes_note__p0001__c001,0.484437
1,Q002,What does MapBiomas indicate about land use an...,MapBiomas,land use and land cover,direct_answer,False,5,1,1,0.333333,MapBiomas; INPE; Project limitations,synthetic_mapbiomas_land_use_note; synthetic_i...,mapbiomas__synthetic_mapbiomas_land_use_note__...,0.574277
2,Q003,How can MapBiomas data help identify native ve...,MapBiomas,native vegetation,explanation,False,5,1,1,0.333333,MapBiomas; INPE; Project limitations,synthetic_mapbiomas_land_use_note; synthetic_i...,mapbiomas__synthetic_mapbiomas_land_use_note__...,0.459135
3,Q004,How can INPE and MapBiomas evidence complement...,MapBiomas; INPE,source comparison,comparison,False,5,1,1,0.666667,MapBiomas; INPE; Project limitations,synthetic_mapbiomas_land_use_note; synthetic_i...,mapbiomas__synthetic_mapbiomas_land_use_note__...,0.691901
4,Q005,What is the difference between deforestation e...,MapBiomas; INPE,methodology,comparison,False,5,1,1,0.666667,INPE; MapBiomas; Project limitations,synthetic_inpe_prodes_note; synthetic_mapbioma...,inpe__synthetic_inpe_prodes_note__p0001__c001,0.976538


## 7. Inspect retrieval failures

This helps identify questions where the expected source was not retrieved.

With synthetic demo data, some failures are expected. With real MapBiomas and INPE documents, this table becomes more meaningful.


In [8]:
if not retrieval_eval.empty:
    retrieval_failures = retrieval_eval[retrieval_eval["source_hit_at_k"] == 0].copy()
    print("Retrieval failures:", len(retrieval_failures))
    display_columns = [
        "question_id",
        "expected_source",
        "retrieved_sources",
        "expected_theme",
        "question",
    ]
    retrieval_failures[display_columns].head(20)
else:
    print("No retrieval evaluation results available.")


Retrieval failures: 2


## 8. Generate answers for golden questions

This section can run in two modes.

### Retrieval-only mode

This is the default. It does not call an LLM and does not require an API key.

### LLM-based mode

Set:

```python
RETRIEVAL_ONLY = False
```

and configure:

```text
OPENAI_API_KEY
```

before running the cell.


In [9]:
from agro_rag.evaluation.evaluate_answers import (
    generate_answers_for_golden_questions,
)

RETRIEVAL_ONLY = True

api_key_available = bool(os.getenv("OPENAI_API_KEY"))

if not RETRIEVAL_ONLY and not api_key_available:
    print("OPENAI_API_KEY not found. Switching to retrieval-only mode.")
    RETRIEVAL_ONLY = True

generated_answers_path = REPORTS_DIR / "rag_generated_answers.csv"

if chroma_is_ready(CHROMA_DB_DIR):
    generated_answers = generate_answers_for_golden_questions(
        golden_questions_df=golden_questions,
        persist_directory=CHROMA_DB_DIR,
        collection_name="agro_environmental_documents",
        n_results=5,
        retrieval_only=RETRIEVAL_ONLY,
    )

    generated_answers.to_csv(generated_answers_path, index=False)
    print("Generated answers saved to:", generated_answers_path)
else:
    generated_answers = pd.DataFrame()
    print("Skipped answer generation because ChromaDB is not available.")

generated_answers.head()


Loading weights: 100%|█████████████████████████████████| 103/103 [00:00<00:00, 6623.94it/s]


Generated answers saved to: C:\Users\aferr\Documents\GitHub\01-rag-document-intelligence\reports\rag_generated_answers.csv


,question_id,question,expected_source,expected_theme,expected_answer_type,unsafe_risk,answer,retrieved_sources,retrieved_documents,n_retrieved_chunks
0,Q001,What does INPE / PRODES indicate about annual ...,INPE,deforestation,direct_answer,False,LLM generation was not executed. Retrieval-onl...,INPE; MapBiomas; Project limitations,synthetic_inpe_prodes_note; synthetic_mapbioma...,3
1,Q002,What does MapBiomas indicate about land use an...,MapBiomas,land use and land cover,direct_answer,False,LLM generation was not executed. Retrieval-onl...,MapBiomas; INPE; Project limitations,synthetic_mapbiomas_land_use_note; synthetic_i...,3
2,Q003,How can MapBiomas data help identify native ve...,MapBiomas,native vegetation,explanation,False,LLM generation was not executed. Retrieval-onl...,MapBiomas; INPE; Project limitations,synthetic_mapbiomas_land_use_note; synthetic_i...,3
3,Q004,How can INPE and MapBiomas evidence complement...,MapBiomas; INPE,source comparison,comparison,False,LLM generation was not executed. Retrieval-onl...,MapBiomas; INPE; Project limitations,synthetic_mapbiomas_land_use_note; synthetic_i...,3
4,Q005,What is the difference between deforestation e...,MapBiomas; INPE,methodology,comparison,False,LLM generation was not executed. Retrieval-onl...,INPE; MapBiomas; Project limitations,synthetic_inpe_prodes_note; synthetic_mapbioma...,3


## 9. Evaluate generated answers

This uses rule-based checks for:

- citation or evidence presence;
- limitation presence;
- cautious language;
- unsafe legal claims;
- basic answer score.

Output:

```text
reports/rag_answer_evaluation.csv
```


In [10]:
from agro_rag.evaluation.evaluate_answers import (
    evaluate_answers_dataframe,
    summarize_answer_evaluation,
)

answer_eval_path = REPORTS_DIR / "rag_answer_evaluation.csv"

if not generated_answers.empty:
    answer_eval = evaluate_answers_dataframe(generated_answers)
    answer_eval.to_csv(answer_eval_path, index=False)

    answer_summary = summarize_answer_evaluation(answer_eval)

    print("Answer evaluation summary:")
    print(answer_summary)
    print("Saved to:", answer_eval_path)
else:
    answer_eval = pd.DataFrame()
    answer_summary = {}
    print("No generated answers available for evaluation.")

answer_eval.head()


Answer evaluation summary:
{'n_answers': 20, 'mean_answer_score': 0.0, 'citation_or_evidence_rate': 0.0, 'limitation_rate': 0.0, 'cautious_language_rate': 0.0, 'unsafe_claim_rate': 0.0}
Saved to: C:\Users\aferr\Documents\GitHub\01-rag-document-intelligence\reports\rag_answer_evaluation.csv


,question_id,question,expected_answer_type,unsafe_risk,answer,answer_length,citation_or_evidence_present,limitation_present,cautious_language_present,unsafe_claim_flag,answer_score
0,Q001,What does INPE / PRODES indicate about annual ...,direct_answer,False,LLM generation was not executed. Retrieval-onl...,62,0,0,0,0,0
1,Q002,What does MapBiomas indicate about land use an...,direct_answer,False,LLM generation was not executed. Retrieval-onl...,62,0,0,0,0,0
2,Q003,How can MapBiomas data help identify native ve...,explanation,False,LLM generation was not executed. Retrieval-onl...,62,0,0,0,0,0
3,Q004,How can INPE and MapBiomas evidence complement...,comparison,False,LLM generation was not executed. Retrieval-onl...,62,0,0,0,0,0
4,Q005,What is the difference between deforestation e...,comparison,False,LLM generation was not executed. Retrieval-onl...,62,0,0,0,0,0


## 10. Inspect unsafe-risk questions

These questions are important because the system must avoid legal overclaims.

Expected behavior:

```text
safe refusal
+ limitation
+ cautious language
```


In [11]:
if not answer_eval.empty:
    unsafe_eval = answer_eval[answer_eval["unsafe_risk"] == True].copy()
    display_columns = [
        "question_id",
        "question",
        "limitation_present",
        "cautious_language_present",
        "unsafe_claim_flag",
        "answer_score",
        "answer",
    ]
    unsafe_eval[display_columns].head(20)
else:
    print("No answer evaluation results available.")


## 11. Create evaluation summary table

This table combines retrieval and answer summary metrics.


In [12]:
summary_records = []

if retrieval_summary:
    summary_records.extend(
        [
            {
                "evaluation_layer": "retrieval",
                "metric": key,
                "value": value,
            }
            for key, value in retrieval_summary.items()
        ]
    )

if answer_summary:
    summary_records.extend(
        [
            {
                "evaluation_layer": "answer",
                "metric": key,
                "value": value,
            }
            for key, value in answer_summary.items()
        ]
    )

summary_df = pd.DataFrame(summary_records)

summary_path = REPORTS_DIR / "rag_evaluation_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("Summary saved to:", summary_path)
summary_df


Summary saved to: C:\Users\aferr\Documents\GitHub\01-rag-document-intelligence\reports\rag_evaluation_summary.csv


,evaluation_layer,metric,value
0,retrieval,n_questions,20.00
1,retrieval,source_hit_rate_at_k,0.90
2,retrieval,mean_precision_at_k,0.45
3,retrieval,mean_recall_at_k,0.90
4,answer,n_answers,20.00
5,answer,mean_answer_score,0.00
6,answer,citation_or_evidence_rate,0.00
7,answer,limitation_rate,0.00
8,answer,cautious_language_rate,0.00
9,answer,unsafe_claim_rate,0.00


## 12. Basic interpretation of evaluation results

Use the following initial targets:

| Metric | Initial target |
|---|---|
| Source hit rate@5 | 80% or higher |
| Mean answer score | 2.0 or higher |
| Unsafe claim rate | 0% |
| Limitation rate for risk answers | 90% or higher |

For the synthetic demo, these targets may not be fully meaningful.

For real evaluation, add real MapBiomas and INPE documents, rebuild the vector store and rerun this notebook.


In [13]:
if not summary_df.empty:
    print("Evaluation summary:")
    for _, row in summary_df.iterrows():
        print(f"- {row['evaluation_layer']} | {row['metric']}: {row['value']}")
else:
    print("No evaluation summary was generated.")


Evaluation summary:
- retrieval | n_questions: 20.0
- retrieval | source_hit_rate_at_k: 0.9
- retrieval | mean_precision_at_k: 0.45
- retrieval | mean_recall_at_k: 0.9
- answer | n_answers: 20.0
- answer | mean_answer_score: 0.0
- answer | citation_or_evidence_rate: 0.0
- answer | limitation_rate: 0.0
- answer | cautious_language_rate: 0.0
- answer | unsafe_claim_rate: 0.0


## 13. Save detailed failure review files

These files help the analyst inspect where the system failed.

Generated files:

```text
reports/rag_retrieval_failures.csv
reports/rag_unsafe_answer_review.csv
```


In [14]:
if not retrieval_eval.empty:
    retrieval_failures_path = REPORTS_DIR / "rag_retrieval_failures.csv"
    retrieval_failures = retrieval_eval[retrieval_eval["source_hit_at_k"] == 0].copy()
    retrieval_failures.to_csv(retrieval_failures_path, index=False)
    print("Retrieval failures saved to:", retrieval_failures_path)

if not answer_eval.empty:
    unsafe_review_path = REPORTS_DIR / "rag_unsafe_answer_review.csv"
    unsafe_review = answer_eval[answer_eval["unsafe_risk"] == True].copy()
    unsafe_review.to_csv(unsafe_review_path, index=False)
    print("Unsafe answer review saved to:", unsafe_review_path)


Retrieval failures saved to: C:\Users\aferr\Documents\GitHub\01-rag-document-intelligence\reports\rag_retrieval_failures.csv
Unsafe answer review saved to: C:\Users\aferr\Documents\GitHub\01-rag-document-intelligence\reports\rag_unsafe_answer_review.csv


## 14. Output files generated

This notebook may generate:

```text
data/processed/rag_eval_questions.csv
reports/rag_retrieval_evaluation.csv
reports/rag_generated_answers.csv
reports/rag_answer_evaluation.csv
reports/rag_evaluation_summary.csv
reports/rag_retrieval_failures.csv
reports/rag_unsafe_answer_review.csv
```


In [15]:
expected_outputs = [
    golden_questions_path,
    retrieval_eval_path,
    generated_answers_path,
    answer_eval_path,
    summary_path,
    REPORTS_DIR / "rag_retrieval_failures.csv",
    REPORTS_DIR / "rag_unsafe_answer_review.csv",
]

print("Expected outputs:")
for path in expected_outputs:
    print("-", path, "| exists:", path.exists())


Expected outputs:
- C:\Users\aferr\Documents\GitHub\01-rag-document-intelligence\data\processed\rag_eval_questions.csv | exists: True
- C:\Users\aferr\Documents\GitHub\01-rag-document-intelligence\reports\rag_retrieval_evaluation.csv | exists: True
- C:\Users\aferr\Documents\GitHub\01-rag-document-intelligence\reports\rag_generated_answers.csv | exists: True
- C:\Users\aferr\Documents\GitHub\01-rag-document-intelligence\reports\rag_answer_evaluation.csv | exists: True
- C:\Users\aferr\Documents\GitHub\01-rag-document-intelligence\reports\rag_evaluation_summary.csv | exists: True
- C:\Users\aferr\Documents\GitHub\01-rag-document-intelligence\reports\rag_retrieval_failures.csv | exists: True
- C:\Users\aferr\Documents\GitHub\01-rag-document-intelligence\reports\rag_unsafe_answer_review.csv | exists: True


## 15. Final interpretation

This notebook completes the first evaluation layer of the project.

The evaluation is intentionally simple, but it checks the most important behaviors for a safe RAG system:

- retrieve evidence from the expected source;
- avoid unsupported answers;
- mention evidence;
- mention limitations;
- avoid unsafe legal claims;
- support human review.

The safest interpretation is:

> The evaluation workflow helps identify whether the RAG system is reliable enough for environmental screening support, but it does not replace expert review.
